# Module 32 — Exercise 2: Bounded Queues and Backpressure Shedding

When request arrival rate exceeds server processing capacity, unbounded in-memory queues cause memory exhaustion (OOM) and latency spikes.

In this exercise, you will implement a Bounded Worker Queue with load shedding and fast rejection under overload.

| Detail | Value |
|---|---|
| **Time** | 35 minutes |
| **Prerequisites** | Module 32 README |



# Your turn


### Task 1: Bounded Queue with Rejection

Implement `BoundedWorkQueue(max_size)`:
- `submit(job)`: If queue has reached `max_size`, reject the job immediately (raising `QueueFullError` or returning `False`). Otherwise, enqueue and return `True`.
- `process_next()`: Dequeue and process next item; if empty, return `None`.


In [ ]:
# ANSWER 1
class QueueFullError(Exception):
    pass

class BoundedWorkQueue:
    def __init__(self, max_size: int = 5):
        self.max_size = max_size
        self.queue: list[object] = []

    def submit(self, job: object) -> bool:
        if len(self.queue) >= self.max_size:
            raise QueueFullError("Queue capacity reached. Shedding load.")
        self.queue.append(job)
        return True

    def process_next(self) -> object | None:
        if not self.queue:
            return None
        return self.queue.pop(0)



## Self-Check Harness


In [ ]:
def check(passed: bool, msg: str) -> bool:
    status = "PASS" if passed else "FAIL"
    print(f"{status}  {msg}")
    return passed

bq = BoundedWorkQueue(max_size=3)
for i in range(3):
    bq.submit(f"job_{i}")

rejected = False
try:
    bq.submit("job_overflow")
except QueueFullError:
    rejected = True

processed = bq.process_next()
bq.submit("job_replacement")

results = [
    check(rejected is True, "Task 1: Overflow request rejected immediately when queue was full"),
    check(processed == "job_0", "Task 1: FIFO queue preserved order"),
    check(len(bq.queue) == 3, "Task 1: Replacement job accepted once space was freed"),
]
print(f"Summary: {sum(results)}/{len(results)} checks passed.")

